<a href="https://colab.research.google.com/github/omairtahir3/flyrank_assignment01/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

I'm picking this lane because the decision it supports is one I can picture a real person acting
on immediately: a content/SEO reviewer opens a ranked queue on Monday morning and works down it,
because they can't look at all 30,000+ pages, only the top of the list. The starter pipeline in
this repo already runs this exact workflow end to end (baseline score -> model -> ranked queue),
so I have committed evidence, not just a hope, that this lane is workable: the baseline rule gets
about 12 of its top 50 picks right, the random forest gets about 37 of 50 right (see Section 3).
That gap is the whole argument for the lane, a simple rule leaves real value on the table, and a
model that combines many tangled signals recovers a lot of it.

I'm keeping Lane 1 (signal analysis) and Lane 3 (clustering) as things I might fold in later as
supporting EDA, but the primary deliverable I want at the end of 7 weeks is a ranked, reason-coded
review queue, not just a report of correlations or a set of cluster names.


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given a client's existing content inventory and its trailing performance
signals, which pages should a content reviewer look at first this week, and why?

**Decision this improves:** which page(s) get a human content review first, out of far more
candidates than the team has time to check by hand.

**Who acts, and what do they do:** a content/SEO reviewer (or a content-ops lead triaging for
their team) works down a ranked queue, opens the top-ranked pages, reads the reason codes
attached to each (e.g. "declining with demand", "stale but visible", "thin but visible"), and
decides whether to refresh, expand, protect, prune, or simply keep monitoring that page. The
review capacity is limited, realistically the team can only act on the top 20-50 pages in a
given cycle, so where a page lands in the ranking is the whole game.

**Cost of a wrong call:**
- *False positive* (page ranked high but isn't actually a problem): a reviewer spends roughly
  20-40 minutes investigating a page that didn't need attention, wasted review hours, and it
  erodes trust in the queue if it happens often.
- *False negative* (a real, high-demand, declining page ranked low or missed): the page keeps
  losing visibility/clicks silently because nobody with limited time ever reached it in the
  queue,  a genuine, avoidable loss of traffic that a first-come rule-based system would have
  missed by never surfacing it.
- Because review capacity is fixed and small relative to inventory size, ranking quality near the
  top of the list matters far more than raw accuracy across the whole 30,000+ rows, which is why
  precision@K (not overall accuracy) is the metric that matches the real decision.

**Why data/ML helps at all:** a single if-statement rule (the starter baseline) already exists and
is transparent, but it only gets ~24% of its top-50 picks right in the starter data. The signals
that actually predict decline, position, CTR, freshness, word count, engagement, trend, all
interacting differently by content type and intent, are individually weak and mutually tangled;
that's exactly the situation where a plain hand-written rule underperforms and a model that can
weigh many correlated, nonlinear signals together earns its place. It's not "train a model" for
its own sake, it's "the rule demonstrably leaves ~3x the precision on the table, and a model
recovers it, on evidence we can already see" (Section 3).


In [ ]:
!git clone https://github.com/omairtahir3/flyrank_assignment01.git

Cloning into 'flyrank_assignment01'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 96 (delta 18), reused 81 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 1.83 MiB | 9.99 MiB/s, done.
Resolving deltas: 100% (18/18), done.


## 3. Quick look at the data (2-3 real numbers)

Loading the starter dataset (`data/raw/content_refresh_anonymized.csv`) and pulling real numbers
that argue for this lane, plus citing the already-committed model comparison from
`outputs/model_report.md` (produced by this repo's own pipeline, not re-run here).


In [ ]:
import pandas as pd

df = pd.read_csv("flyrank_assignment01/data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_clients = df["client_id"].nunique()

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
trend_counts = df["trend_direction"].value_counts()

print(f"Rows: {n_rows:,} across {n_clients} pseudonymized clients")
print()
print(f"'declining_with_demand' (trend_direction == down AND impressions_90d >= 100): "
      f"{declining_with_demand:,} rows ({declining_with_demand / n_rows:.1%} of all rows)")
print(f"'stale_visible_page' (no update in 180+ days AND impressions_90d >= 500): "
      f"{stale_visible:,} rows ({stale_visible / n_rows:.1%} of all rows)")
print()
print("trend_direction breakdown:")
print(trend_counts)


Rows: 30,000 across 32 pseudonymized clients

'declining_with_demand' (trend_direction == down AND impressions_90d >= 100): 13,152 rows (43.8% of all rows)
'stale_visible_page' (no update in 180+ days AND impressions_90d >= 500): 17 rows (0.1% of all rows)

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

**What I can claim, with evidence:**
- Observed associations between trailing-90-day signals (position, CTR, freshness, engagement,
  trend) and whether a page later carries a "declining" label, measured on this starter slice with
  client-holdout validation (whole clients held out of training, not just rows).
- That a simple, transparent baseline rule is beatable: the committed pipeline output shows the
  baseline rule reaching precision@50 = 0.240 versus the random forest's 0.740 (roughly 12 vs 37
  correct out of the top 50 recommendations), a decision-support improvement, not a guarantee.
- A ranked, reason-coded queue that helps a reviewer spend limited time on the most promising
  candidates first, i.e. decision-support, not an automated verdict.

**What I will not claim:**
- That a refresh *causes* a page to recover, that needs an experiment or a causal design, which
  this observational data cannot provide on its own.
- Any claim about Google's ranking algorithm, AI citations, or AI search visibility, the data
  only contains observed sessions/impressions/clicks, never the mechanics behind them.
- That results validated on this 30,000-row starter slice automatically hold on the ~79M-row
  warehouse, if I move to the warehouse, the result has to be re-earned there with its own
  validation.
- That the current label (`trend_direction == "down"`, a same-window bucket) is the ideal target
  it's a beginner proxy per the lane guide; a stronger version of this lane defines a genuinely
  future-looking label (prior 90 days of features -> next 30 days of outcome) before the capstone.
